# The Real $150,000 Was the Friends We Made Along the Way

**This version** submits the best public portfolio we know of, [SomeJoint's 21.98 notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi), **unchanged and fully credited**, and shows how we checked it before submitting.

**How the score is produced, in two steps:**
1. **Our check (this notebook).** We run the unchanged official verifiers on every circuit using our memory-lean [kickmix fork](https://github.com/manwithacat/kickmix-macos), loaded through our [`kickmix_tools` utility script](https://www.kaggle.com/code/manwithacat/kickmix-tools). The fork gives the same results as the organisers' build with far less memory. The numbers printed below are *our* result.
2. **The official score.** When the `submission.zip` written here is submitted, Kaggle's grader runs the **official** verifier and kickmix build on it, with its own samples. That is the only score that counts; ours is a preview.

Why this notebook? Earlier versions built our **classical-mask projection** for DQI (9.13909, explained at the end). That projection now sits in the lineage of the 21.98 DQI decoder, credited by its authors, alongside many others' work. Hence the title.

| Task | Circuit by | Q | Mean Toffolis | Official cost (local, default seed) |
|---|---|---:|---:|---:|
| invsqrt | mrmeows | 63 | 1,348.5 | 103,546 |
| DQI | SomeJoint (Agniv Banerjee) and GPT-6 Astra Pro, on Ahmed Berat Özer, Alex Chilton and jday96314 | 883 | 467,789.6 | 4.36484e+11 |
| Chemistry | busyaprime, on Ahmed Berat Özer's v9 and its credited chain | 431 | 17,330.0 | 3.55065e+08 |

Full credits are in the last section and, in more detail, in [the source notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi). We claim none of these circuits as our own work. Code in this notebook is AI-assisted (Claude), Apache-2.0.

## Inputs, pinned by SHA-256

Attached: the competition data, the output of [somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi](https://www.kaggle.com/code/somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi), our utility script [manwithacat/kickmix-tools](https://www.kaggle.com/code/manwithacat/kickmix-tools) and the wheel dataset [manwithacat/kickmix-fork-wheels](https://www.kaggle.com/datasets/manwithacat/kickmix-fork-wheels). Internet is off. If the upstream output changes, the hash check below stops the run instead of submitting something we haven't checked.

In [1]:
import hashlib, json, shutil, sys, time, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
ARCHIVE_SHA = "be8301da5551ab72edc6b92ddbe1cefa0751164d845b5adb9d30856f5695e0d4"
MEMBERS = {"dfthcsos/angle_k_0.kmb": "0942f5305a87c5a4fcf6651e62df9101cec4671415f169a71de60ba9ea75ff7c", "dfthcsos/angle_k_1.kmb": "54951c2665fac7bf59aac7c4889f6b3785fe080951b9914168d86a498cee7b49", "dfthcsos/angle_k_10.kmb": "2c7006937399980a5d4642acbbd7033dbc1517eca941cd84eb86c752c66c6120", "dfthcsos/angle_k_11.kmb": "d4342edd711b9bea1afd9ef79d931bfbd1845078b45b65648273fb033d225f50", "dfthcsos/angle_k_12.kmb": "785565ad2fbd99b53b89931795ceb9c05e7b1524c7f1a7f58ffbf6baa5e0d8b5", "dfthcsos/angle_k_13.kmb": "c265d77603a919b83c1d20ced3252692a5f902b0175c6cfdbac7ce584bd1f9c2", "dfthcsos/angle_k_14.kmb": "c9df4f1975b63ed9014ecffe693337c6d6919d81a7b32025e78dbdc664486881", "dfthcsos/angle_k_2.kmb": "c618f287eb496c1845b7200b35f0fdd0491f1184f3b09f118a7d421234c8a14d", "dfthcsos/angle_k_3.kmb": "f091a367d27bdf2a721122ec3515784a7d0513c869e7b0c452db147798197599", "dfthcsos/angle_k_4.kmb": "945ad68ef9be3e6c4f3e30562ffe96399af8e51ade7877e3e9dc9d8b870154b5", "dfthcsos/angle_k_5.kmb": "47a56f9bf5c95b0e2006b390ba868f874d439c594d16abc729d19c2b1d894928", "dfthcsos/angle_k_6.kmb": "85f4675c9b853e8714aff1af5448b4e8321a5cdf6087d0105ee57c3bdcc306d1", "dfthcsos/angle_k_7.kmb": "1946e4372780fbb04da292d3544d206a6676b72a38546c461df813761e67225b", "dfthcsos/angle_k_8.kmb": "95c635076a3e387d967715bf6fc203466aaf0de654cef8bc550892f7ba7ca35e", "dfthcsos/angle_k_9.kmb": "7a8338e7ef8a04718c07998539b8d536245bf38e1844599a13219a45f57097b4", "dfthcsos/submission.npz": "a60e21cb380474976b7d99dbf97889beb0784928d3a6c0becef12bc982496ab6", "dqi.kmb": "ef334fee63f3367fe721549dd94d4264793c56e3c49a09177c05b353ac71c927", "invsqrt.kmb": "0766a111a76f241aa57fbbed466436d743be13436943ddb0628c291ca07d67d6"}
WHEEL_NAME = "kickmix-0.2.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def find(name, root=INPUT):
    hits = sorted(Path(root).rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits

source_zip = next((p for p in find("submission.zip") if sha256(p) == ARCHIVE_SHA), None)
assert source_zip is not None, "no attached submission.zip matches the pinned SHA-256: upstream changed, review before submitting"
SUB = WORK / "portfolio"
shutil.rmtree(SUB, ignore_errors=True)
with zipfile.ZipFile(source_zip) as z:
    names = sorted(i.filename for i in z.infolist() if not i.is_dir())
    assert names == sorted(MEMBERS), names
    z.extractall(SUB)
for name, digest in MEMBERS.items():
    assert sha256(SUB / name) == digest, f"member hash mismatch: {name}"
print(f"{source_zip}: archive and all {len(MEMBERS)} members match the pinned hashes")

/kaggle/input/notebooks/somejoint/quantum-circuits-8-95-lb-cost-breakdown/submission.zip: archive and all 18 members match the pinned hashes


## Load the utility script and install the fork offline

`kickmix_tools` installs the fork into its own directory from the attached wheel, after checking the wheel against the release's `SHA256SUMS`. The competition's `kickmix.so` is never touched: each verification runs in a fresh subprocess with exactly one build on its import path.

In [2]:
tool_dirs = [p.parent for root in ("/kaggle/usr/lib", "/kaggle/input") if Path(root).is_dir()
             for p in Path(root).rglob("kickmix_tools.py")]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools from", tool_dirs[0])

wheel = find(WHEEL_NAME)[0]
sums = (wheel.parent / "SHA256SUMS").read_text()
expected = next(line.split()[0] for line in sums.splitlines() if line.rstrip().endswith(WHEEL_NAME))
assert sha256(wheel) == expected, "wheel does not match the release SHA256SUMS"
FORK_DIR = kt.install(str(wheel))
print("competition data:", kt.competition_dir())

kickmix_tools from /kaggle/usr/lib/kickmix-tools
fork: installed via local wheel kickmix-0.2.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
fork: directory /kaggle/working/kickmix_fork
competition data: /kaggle/input/competitions/google-verifiable-quantum-circuit-compilation


## Our check: the official verifiers on the fork

Full official settings: default shot counts, the seed derived from each circuit's SHA-256, empty salt. Chemistry takes well under a minute, DQI a few minutes, invsqrt the longest.

In [3]:
CIRCUITS = {"dfthcsos": SUB / "dfthcsos", "dqi": SUB / "dqi.kmb", "invsqrt": SUB / "invsqrt.kmb"}
results = {}
for task, circuit in CIRCUITS.items():
    r = kt.verify(task, circuit, build="fork")
    results[task] = r
    res = r["result"]
    mem = r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"]
    extra = f", errors {res['value_errors']} value / {res['phase_errors']} phase" if task == "dqi" else ""
    print(f"{task:9} {res['status']}  cost {res['score']:.6e}  Q {res['num_qubits']}{extra}  "
          f"{r['wall_seconds']:.0f} s, peak {mem:,.0f} MB")
assert all(r["result"]["status"] == "PASS" for r in results.values()), "a verifier did not pass"

dfthcsos  PASS  cost 3.550651e+08  Q 431  29 s, peak 729 MB
dqi       PASS  cost 4.364844e+11  Q 883, errors 0 value / 1 phase  97 s, peak 2,092 MB
invsqrt   PASS  cost 1.035462e+05  Q 63  480 s, peak 473 MB


In [4]:
from math import prod
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "dfthcsos": 3355599588.3155375}
ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
for t, x in ratios.items():
    print(f"{t:9} {x:8.4f}x cheaper than the reference")
gm = prod(ratios.values()) ** (1 / 3)
print(f"our local geometric mean: {gm:.5f}  (the leaderboard re-measures with its own samples)")

invsqrt    20.0565x cheaper than the reference
dqi        55.9916x cheaper than the reference
dfthcsos    9.4507x cheaper than the reference
our local geometric mean: 21.97591  (the leaderboard re-measures with its own samples)


## Write the submission

The archive is the attached one, byte for byte, so what Kaggle grades is exactly what was checked above.

In [5]:
shutil.copyfile(source_zip, WORK / "submission.zip")
assert sha256(WORK / "submission.zip") == ARCHIVE_SHA
shutil.rmtree(SUB)
(WORK / "check.json").write_text(json.dumps({
    "archive_sha256": ARCHIVE_SHA, "geometric_mean_local": gm, "fork_install": kt._STATE.get("fork_how"),
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
print("wrote", WORK / "submission.zip")

wrote /kaggle/working/submission.zip


## Credits

All three circuits are other people's public work, used unchanged; see [the 21.98 notebook](https://www.kaggle.com/code/somejoint/quantum-circuits-21-98-local-gm-883-qubit-dqi) for the full chains and licences.

- **invsqrt:** **mrmeows**, the 63-qubit hybrid build from the Apache-2.0 dataset [qcc-1375-open-circuits](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits), on the public line of **Akmal Xodarev** (busyaprime), **Phuc Thai** (DePie), **Dmitrii Gluzdov** and **marsyas**.
- **DQI:** **Agniv Banerjee (SomeJoint)** and team, with **GPT-6 Astra Pro**'s 658-Toffoli jet; **Ahmed Berat Özer**'s narrowing (v9); **Alex Chilton**'s 883-qubit Schur decoder design, based on **jday96314**'s decoder; the public mask projection of **mrmeows** and **Ahmed Berat Özer**, after ours; earlier links by **Dmitrii Gluzdov**, **ViberTeam** and **Phuc Thai (DePie)**.
- **Chemistry:** **busyaprime (Akmal Xodarev)**'s refitted weights on **Ahmed Berat Özer**'s version 9 circuits, crediting **Michael Rustad**, **mrmeows**, **heliosli**, **Vladislav Lavrentev (lightningv08)**, **marculera**, **kozykappa**, **alex chilton** and **Evgen Dvorkin**.
- **Tooling:** kickmix and the verifiers by **Google Quantum AI** (their own terms); the fork and `kickmix_tools` by manwithacat, AI-assisted (Claude).

If we have missed anyone, tell us in the comments and we will add them.

## Background: our classical-mask projection (earlier versions, 9.13909)

The DQI task supplies 70 field syndromes of an error vector over $\mathrm{GF}(2^{12})$. The target phase is a parity of bits in that hidden vector, selected by *classical* masks $c_j$ from a measurement record. Writing the syndromes as $s_i=\sum_j e_j\alpha^{(i+1)j}$, we can choose 70 classical field words $\lambda_i$ so that the known phase $\operatorname{Tr}(\sum_i\lambda_i s_i)$ accounts for the masks at positions $j=0,\ldots,69$ (a $70\times70$ Vandermonde system). Subtracting this known linear phase leaves residual masks that are zero at those 70 positions, so the decoder's scan skips its quantum work there. On SomeJoint's earlier decoder this saved 5,559 mean Toffolis (9.12718 → 9.13909). It is a compilation step that uses classical side information, not a faster decoding theorem.

The full build and official check are in the earlier versions of this notebook (version history).